# Fase 3: CNN Propia (MEJORADA)
## Sistema Inteligente para la Detección de Tumores Cerebrales en MRI

Arquitectura CNN mejorada: 4 bloques convolucionales (2 conv + BN + ReLU + MaxPool por bloque)
con dropout reducido y he_normal initialization.

In [ ]:
import sys
import os
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf

sys.path.append(os.path.abspath(".."))
from src.utils.config import (
    TRAINING_DIR, TESTING_DIR, CLASSES,
    IMG_SIZE, BATCH_SIZE, EPOCHS, MODELS_DIR
)
from src.data.loader import load_image_paths, load_and_preprocess_image
from src.data.preprocessing import encode_labels, split_dataset
from src.data.augmentation import get_tf_generators
from src.models.cnn_custom import build_cnn_custom
from src.models.train import train_model
from src.evaluation.metrics import evaluate_model
from src.evaluation.plots import plot_training_history, plot_confusion_matrix, plot_per_class_metrics

PROJECT_DIR = os.path.abspath(os.path.join(os.getcwd(), '..'))
DOCS_DIR = os.path.join(PROJECT_DIR, 'docs')
os.makedirs(MODELS_DIR, exist_ok=True)
os.makedirs(DOCS_DIR, exist_ok=True)

%matplotlib inline

In [ ]:
print("Cargando Training (entrenamiento + validación)...")
train_paths, train_labels = load_image_paths(TRAINING_DIR)
X_train_full = []
y_train_full = []
for path, label in zip(train_paths, train_labels):
    img = load_and_preprocess_image(path, IMG_SIZE)
    if img is not None:
        X_train_full.append(img)
        y_train_full.append(label)
X_train_full = np.array(X_train_full)
y_train_full = encode_labels(np.array(y_train_full))
print(f"Training cargado: {X_train_full.shape}")
print(f"Distribución: {np.bincount(y_train_full)}")

print("\nCargando Testing (evaluación final)...")
test_paths, test_labels = load_image_paths(TESTING_DIR)
X_test_final = []
y_test_final = []
for path, label in zip(test_paths, test_labels):
    img = load_and_preprocess_image(path, IMG_SIZE)
    if img is not None:
        X_test_final.append(img)
        y_test_final.append(label)
X_test_final = np.array(X_test_final)
y_test_final = encode_labels(np.array(y_test_final))
print(f"Testing cargado: {X_test_final.shape}")
print(f"Distribución: {np.bincount(y_test_final)}")

In [ ]:
X_train, X_val, _, y_train, y_val, _ = split_dataset(X_train_full, y_train_full)
train_ds, val_ds, _ = get_tf_generators(
    X_train, X_val, X_test_final, y_train, y_val, y_test_final
)
print(f"Train: {X_train.shape[0]}, Val: {X_val.shape[0]}, Test final: {X_test_final.shape[0]}")

In [ ]:
model = build_cnn_custom()
model.summary()

try:
    tf.keras.utils.plot_model(model, os.path.join(MODELS_DIR, 'cnn_custom_architecture.png'), show_shapes=True)
except Exception as e:
    print(f"[WARNING] No se pudo generar el diagrama del modelo (requiere GraphViz): {e}")

In [ ]:
history = train_model(model, train_ds, val_ds, epochs=EPOCHS, train_size=len(X_train))

In [ ]:
plot_training_history(history, os.path.join(DOCS_DIR, 'cnn_propia_curvas.png'))

In [ ]:
global_metrics, per_class, cm = evaluate_model(model, X_test_final, y_test_final)

print("=" * 60)
print("MÉTRICAS GLOBALES")
print(f"Accuracy:  {global_metrics['accuracy']:.4f}")
print(f"Precision: {global_metrics['precision']:.4f}")
print(f"Recall:    {global_metrics['recall']:.4f}")
print(f"F1-Score:  {global_metrics['f1_score']:.4f}")

print("\nMÉTRICAS POR CLASE")
for cls, metrics in per_class.items():
    print(f"{cls:>12}: Precision={metrics['precision']:.4f}, Recall={metrics['recall']:.4f}, F1={metrics['f1_score']:.4f}")

In [ ]:
plot_confusion_matrix(cm, os.path.join(DOCS_DIR, 'cnn_propia_confusion.png'))
plot_per_class_metrics(per_class, os.path.join(DOCS_DIR, 'cnn_propia_metrics.png'))

In [ ]:
model.save(os.path.join(MODELS_DIR, 'cnn_custom.keras'))
print("Modelo guardado en models/cnn_custom.keras")